In [1]:
import json
import time
import requests
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np

from bot_template import BaseBot, OrderBook, Order, OrderRequest, OrderResponse, Trade, Side, Product

In [4]:
LONDON_LAT, LONDON_LON = 51.5074, -0.1278

def get_weather(past_steps=96, forecast_steps=96):
    """15-min weather for London. 96 steps = 24 hours.

    Returns DataFrame with: time, temperature, wind_speed, humidity,
    precipitation, cloud_cover, visibility, apparent_temperature.
    """
    variables = "temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,wind_speed_10m,cloud_cover,visibility"
    resp = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": LONDON_LAT, "longitude": LONDON_LON,
        "minutely_15": variables,
        "past_minutely_15": past_steps,
        "forecast_minutely_15": forecast_steps,
        "timezone": "Europe/London",
    })
    resp.raise_for_status()
    m = resp.json()["minutely_15"]
    return pd.DataFrame({
        "time": pd.to_datetime(m["time"]).tz_localize("Europe/London"),
        "temperature": m["temperature_2m"],
        "apparent_temperature": m["apparent_temperature"],
        "humidity": m["relative_humidity_2m"],
        "precipitation": m["precipitation"],
        "wind_speed": m["wind_speed_10m"],
        "cloud_cover": m["cloud_cover"],
        "visibility": m["visibility"],
    })

df_weather = get_weather()
print(f"{len(df_weather)} readings, {df_weather.time.min()} -> {df_weather.time.max()}")
df_weather.tail(5)



192 readings, 2026-02-27 15:15:00+00:00 -> 2026-03-01 15:00:00+00:00


,time,temperature,apparent_temperature,humidity,precipitation,wind_speed,cloud_cover,visibility
187,2026-03-01 14:00:00+00:00,11.6,8.6,82,0.0,18.4,100,16860.0
188,2026-03-01 14:15:00+00:00,11.8,8.7,80,0.0,18.7,100,17360.0
189,2026-03-01 14:30:00+00:00,11.9,8.8,79,0.0,18.7,100,17860.0
190,2026-03-01 14:45:00+00:00,12.1,8.9,77,0.0,18.7,100,18340.0
191,2026-03-01 15:00:00+00:00,12.1,8.9,77,0.0,18.7,100,18840.0


In [5]:
import pandas as pd

# 1. Convert Celsius to Fahrenheit
df_weather['temp_F'] = (df_weather['temperature'] * 9/5) + 32

# 2. Calculate the base metric (temp_F x humidity)
df_weather['wx_metric'] = df_weather['temp_F'] * df_weather['humidity']

# 3. Define the current 12pm-to-12pm session window
# Note: Since we are currently at Feb 28, 2026 ~2:30 PM, the active session is:
session_start = pd.to_datetime("2026-02-28 12:00:00").tz_localize("Europe/London")
session_end   = pd.to_datetime("2026-03-01 12:00:00").tz_localize("Europe/London")

# Filter the DataFrame to only include the current session window
session_df = df_weather[(df_weather['time'] > session_start) & (df_weather['time'] <= session_end)]

# 4. Calculate WX_SPOT (Value exactly at 12 PM at the end of the session)
spot_row = session_df[session_df['time'] == session_end]
wx_spot_fair_value = spot_row['wx_metric'].iloc[0] if not spot_row.empty else None

# 5. Calculate WX_SUM (15-min aggregate / 100)
# Note: Assuming the aggregate is the sum of the wx_metric over the session window. 
wx_sum_fair_value = session_df['wx_metric'].sum() / 100

print(f"Fair Value WX_SPOT: {wx_spot_fair_value}")
print(f"Fair Value WX_SUM:  {wx_sum_fair_value}")

Fair Value WX_SPOT: 4428.3
Fair Value WX_SUM:  3314.9588
